# 003 — Hierarchical Parent-Child Chunking
**Ingestion Series** | Retrieve small, return big

**Problem:** Large chunks = more context but worse retrieval accuracy.
Small chunks = precise retrieval but missing context.

**Solution:** Index small child chunks for retrieval, but return their large parent chunk to the LLM.


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile
✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)


In [3]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


✓ 5 dataset files loaded
  File                    Chars   ~Words
  ----------------------------------------
  ai.txt                  6,221      943
  machine_learning.txt    5,953      870
  nlp.txt                 5,160      693
  deep_learning.txt       5,114      686
  rag.txt                 5,146      715
  ----------------------------------------
  TOTAL                  27,602    3,907


In [4]:
# ── Setup parent and child splitters ─────────────────────────────────────
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"   # silence ChromaDB telemetry noise

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma
import chromadb

# Parent chunks: large (retain full context for generation)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)

# Child chunks: small (precise, clean retrieval)
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=300,  chunk_overlap=30)

# Build parent docs from AI article
parent_chunks = parent_splitter.split_text(ai_text[:15000])
print(f"Parent chunks: {len(parent_chunks)}  avg={sum(len(c) for c in parent_chunks)//len(parent_chunks)} chars")

child_docs = []
for pid, parent in enumerate(parent_chunks):
    for child in child_splitter.split_text(parent):
        child_docs.append(Document(
            page_content=child,
            metadata={"parent_id": pid, "parent_text": parent}
        ))

print(f"Child chunks:  {len(child_docs)}  avg={sum(len(d.page_content) for d in child_docs)//len(child_docs)} chars")
print(f"\nOn average each parent has {len(child_docs)//len(parent_chunks)} children")


Parent chunks: 5  avg=1260 chars
Child chunks:  37  avg=168 chars

On average each parent has 7 children


In [5]:
# ── Build child index (what we search) ───────────────────────────────────
child_vectorstore = Chroma(
    collection_name="child_chunks",
    embedding_function=embeddings,
    client=chromadb.Client()
)
child_vectorstore.add_documents(child_docs)
print(f"✓ Indexed {len(child_docs)} child chunks in ChromaDB")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✓ Indexed 37 child chunks in ChromaDB


In [6]:
# ── Parent-returning retriever ────────────────────────────────────────────
def parent_retriever(query: str, k: int = 3):
    '''Retrieve children, deduplicate by parent_id, return parent texts.'''
    hits = child_vectorstore.similarity_search(query, k=k * 3)
    seen, parents = set(), []
    for hit in hits:
        pid = hit.metadata["parent_id"]
        if pid not in seen:
            seen.add(pid)
            parents.append({
                "parent_id":   pid,
                "parent_text": hit.metadata["parent_text"],
                "child_match": hit.page_content,
            })
        if len(parents) >= k:
            break
    return parents

query = "What are neural networks and how are they used in AI?"
results = parent_retriever(query, k=2)

print(f"Query: {query!r}\n")
for r in results:
    print(f"── Parent {r['parent_id']} ({len(r['parent_text'])} chars) ──")
    print(f"  Matched child: {r['child_match'][:120]}...")
    print(f"  Full parent  : {r['parent_text'][:300]}...")
    print()


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: 'What are neural networks and how are they used in AI?'

── Parent 3 (1191 chars) ──
  Matched child: An artificial neural network (ANN) is modeled loosely after the human brain that is designed to
recognize patterns. They...
  Full parent  : Neural Networks and Deep Learning

An artificial neural network (ANN) is modeled loosely after the human brain that is designed to
recognize patterns. They interpret sensory data through a kind of machine perception, labeling
or clustering raw input. The patterns they recognize are numerical, contai...

── Parent 2 (1487 chars) ──
  Matched child: Neural Networks and Deep Learning...
  Full parent  : Deep Learning Revolution (2010s): The field was transformed by the success of deep learning,
a subset of machine learning. Deep learning uses artificial neural networks with many layers
to learn representations of data. The success of AlexNet in the 2012 ImageNet competition
demonstrated that deep c...



In [7]:
# ── Compare: child-only vs parent-child ──────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def rag_answer(context_text, question):
    chain = (
        ChatPromptTemplate.from_template(
            "Answer using only the context below.\nContext: {ctx}\nQuestion: {q}\nAnswer:"
        )
        | llm | StrOutputParser()
    )
    return chain.invoke({"ctx": context_text[:3000], "q": question})

question = "How do deep learning systems learn from data?"

# Child-only context (small, may miss surrounding detail)
child_hits = child_vectorstore.similarity_search(question, k=3)
child_ctx  = "\n\n".join(h.page_content for h in child_hits)

# Parent context (full surrounding passage)
parent_hits = parent_retriever(question, k=2)
parent_ctx  = "\n\n".join(r["parent_text"] for r in parent_hits)

print("=== CHILD-ONLY CONTEXT (RAG answer) ===")
print(rag_answer(child_ctx, question))
print()
print("=== PARENT-CHILD CONTEXT (RAG answer) ===")
print(rag_answer(parent_ctx, question))


=== CHILD-ONLY CONTEXT (RAG answer) ===
Deep learning systems learn from data through representation learning, which is based on artificial neural networks.

=== PARENT-CHILD CONTEXT (RAG answer) ===


Deep learning systems learn from data by using artificial neural networks with many layers to learn representations of data.


## Key Takeaways
- Small child chunks → **precise vector search** (less noise)
- Large parent chunks → **rich context** for the LLM
- The trade-off: parent context is larger → more tokens → higher cost
- LangChain's `ParentDocumentRetriever` automates this pattern with an `InMemoryStore` or Redis docstore
